# Aube T1: fine-grid CSI

CSI comparison on a common 20 m grid. Fine and regular samples are paired through their stored `ts`.

In [ ]:
from pathlib import Path
import math
import pickle
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from omegaconf import OmegaConf


def find_project_root():
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (path / "python").exists() and (path / "bin").exists():
            return path
    raise RuntimeError("Project root not found.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from modulus.launch.utils import load_checkpoint
from python.CustomMeshGraphNet import MeshGraphNetWithSourceNodes
from python.create_dgl_dataset import TelemacDatasetWithSourceNodes, unpack_dynamic_sample
from python.python_code.data_manip.extraction.telemac_file import TelemacFile
from python.python_code.data_manip.formats.regular_grid import interpolate_on_grid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# User parameters
CONFIG_DIR = PROJECT_ROOT / "bin/conf/research"

# Replace these two paths with the SLURM .out or train .log files.
METHOD_SPECS = [
    {
        "name": "Regular 40 m",
        "config": CONFIG_DIR / "config_overfit_Aube_T1_ghost_40m_useQ.yaml",
        "loss_log": Path("/path/to/regular_40m_training.log"),
        "color": "#1f77b4",
    },
    {
        "name": "Multimesh 40-80-160 m",
        "config": CONFIG_DIR / "config_overfit_Aube_T1_ghost_40m_useQ_multi.yaml",
        "loss_log": Path("/path/to/multimesh_40_160_training.log"),
        "color": "#d95f02",
    },
]

EVENTS = [
    {
        "name": "Q5 overfit",
        "regular_dynamic": "/work/m24046/m24046mrcr/Test_Aube_T1_regular/dataset_40m_ghost/regular_short/08_T1_V9_Topo3_KV5_Q5_0_47-76_interpolated.pkl",
        "fine_dynamic": "/work/m24046/m24046mrcr/Test_Aube_T1_regular/dataset_40m_ghost/fine_dataset/08_T1_V9_Topo3_KV5_Q5_0_0-112.pkl",
        "hydro": "/work/m24046/m24046mrcr/Test_Aube_T1_regular/original_dataset/T1_Q5_V1.liq",
    },
]

PLOT_METHOD_NAMES = ["Regular 40 m", "Multimesh 40-80-160 m"]
REGULAR_MESH_PATH = "/work/m24046/m24046mrcr/Test_Aube_T1_regular/dataset_40m/Aube_regular_T1_40m.slf"
FINE_MESH_PATH = "/work/m24046/m24046mrcr/Test_Aube_T1_regular/original_dataset/T1_V9_Topo3_KV5.slf"

DT_SECONDS = 7200.0
MAX_HOURS = 12
THRESHOLD_M = 0.05
GRID_STEP_M = 20.0
MAX_SEQUENCES = 205
OVERLAP = 1
CHECKPOINT_EVERY = 10

QUAL_EVENT_NAME = "Q5 overfit"
QUAL_SEQUENCE_INDEX = 0
QUAL_HOUR = 12

In [ ]:
LOSS_PATTERN = re.compile(r"epoch:\s*(\d+).*?loss:\s*([0-9.eE+-]+)")


def best_saved_checkpoint(log_path):
    matches = LOSS_PATTERN.findall(Path(log_path).read_text(errors="replace"))
    saved_losses = [
        (int(epoch), float(loss))
        for epoch, loss in matches
        if int(epoch) % CHECKPOINT_EVERY == 0
    ]
    return min(saved_losses, key=lambda item: item[1])


def load_method(spec):
    epoch, loss = best_saved_checkpoint(spec["loss_log"])
    method = dict(spec)
    method["cfg"] = OmegaConf.load(spec["config"])
    method["epoch"] = epoch
    method["loss"] = loss
    return method


METHODS = [load_method(spec) for spec in METHOD_SPECS]
METHOD_BY_NAME = {method["name"]: method for method in METHODS}

for method in METHODS:
    print(f'{method["name"]}: epoch={method["epoch"]}, training loss={method["loss"]:.6e}')

In [ ]:
MAX_STEPS = int(MAX_HOURS * 3600.0 // DT_SECONDS)
HORIZONS_STEPS = list(range(1, MAX_STEPS + 1))
HOURS = np.asarray(HORIZONS_STEPS, dtype=float) * DT_SECONDS / 3600.0
SEQUENCE_LENGTH = MAX_STEPS + 1
QUAL_STEP = int(QUAL_HOUR * 3600.0 / DT_SECONDS)

print(f"steps={HORIZONS_STEPS} | hours={HOURS.tolist()} | sequence_length={SEQUENCE_LENGTH}")

In [ ]:
def load_samples(path):
    with open(path, "rb") as handle:
        return pickle.load(handle)


def build_dataset(method, event, sequence_length):
    cfg = method["cfg"]
    return TelemacDatasetWithSourceNodes(
        name=f'eval_{event["name"]}',
        data_dir=str(cfg.data_dir),
        dynamic_data_files=[event["regular_dynamic"]],
        hydro_data_files=[event["hydro"]],
        inlet_node_lists=OmegaConf.to_container(cfg.inlet_node_lists),
        use_q_feature=bool(cfg.use_q_feature),
        split="test",
        ckpt_path=str(cfg.ckpt_path),
        normalize=True,
        sequence_length=sequence_length,
        overlap=OVERLAP,
        dt_seconds=float(cfg.dt_seconds),
    )


def aligned_fine_sequences(event, sequence_length):
    regular_samples = load_samples(event["regular_dynamic"])
    fine_samples = load_samples(event["fine_dynamic"])
    fine_by_ts = {unpack_dynamic_sample(sample)[2]: sample for sample in fine_samples}
    stride = max(1, sequence_length - OVERLAP)
    starts = range(0, len(regular_samples) - sequence_length + 1, stride)
    return [
        [fine_by_ts[unpack_dynamic_sample(regular_samples[index])[2]] for index in range(start, start + sequence_length)]
        for start in starts
    ]


def build_model(method, dataset):
    cfg = method["cfg"]
    model = MeshGraphNetWithSourceNodes(
        input_dim_nodes_phys=dataset.base_graph.ndata["static"].shape[1] + dataset.physical_dynamic_dim,
        input_dim_nodes_src=len(dataset.source_feature_names),
        input_dim_edges=int(cfg.num_edge_features),
        output_dim=int(cfg.num_output_features),
        processor_size=int(cfg.mp_layers),
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=bool(cfg.do_concat_trick),
        num_processor_checkpoint_segments=int(cfg.num_processor_checkpoint_segments),
    )
    load_checkpoint(str(cfg.ckpt_path), models=model, device=device, epoch=method["epoch"])
    model.to(device).eval()
    return model


def build_context(dataset):
    stats = dataset.node_stats
    return {
        "static_dim": dataset.base_graph.ndata["static"].shape[1],
        "physical_dynamic_dim": dataset.physical_dynamic_dim,
        "state_mean": torch.tensor([stats["h"].item(), stats["u"].item(), stats["v"].item()], device=device),
        "state_std": torch.tensor([stats["h_std"].item(), stats["u_std"].item(), stats["v_std"].item()], device=device),
        "delta_mean": torch.tensor([stats["delta_h"].item(), stats["delta_u"].item(), stats["delta_v"].item()], device=device),
        "delta_std": torch.tensor([stats["delta_h_std"].item(), stats["delta_u_std"].item(), stats["delta_v_std"].item()], device=device),
    }


def rollout_sequence(model, dataset, sequence_index):
    sequence = dataset[sequence_index]
    context = build_context(dataset)
    state = {
        "graph": sequence[0]["graph"].to(device),
        "x_phys": sequence[0]["x_phys"].to(device),
        "x_src": sequence[0]["x_src"].to(device),
    }
    predictions = []

    for next_step in sequence[1:]:
        with torch.no_grad():
            y_pred_n = model(
                graph=state["graph"],
                x_phys=state["x_phys"],
                x_src=state["x_src"],
                edge_features=state["graph"].edata["x"].to(device),
            )

        start = context["static_dim"]
        x_t = state["x_phys"][:, start:start + 3] * context["state_std"] + context["state_mean"]
        y_pred = y_pred_n * context["delta_std"] + context["delta_mean"]
        x_pred = x_t + y_pred
        x_gt_n = next_step["x_phys"][:, start:start + 3].to(device)
        x_gt = x_gt_n * context["state_std"] + context["state_mean"]

        h_mask = (state["x_phys"][:, :4] == torch.tensor([0, 1, 0, 0], device=device)).all(dim=1)
        x_pred = x_pred.clone()
        x_pred[h_mask, 0] = x_gt[h_mask, 0]
        predictions.append(x_pred.cpu().numpy())

        x_next_n = (x_pred - context["state_mean"]) / (context["state_std"] + 1e-12)
        static = state["x_phys"][:, :start]
        forcing = next_step["x_phys"][:, start + 3:start + context["physical_dynamic_dim"]].to(device)
        state = {
            "graph": next_step["graph"].to(device),
            "x_phys": torch.cat((static, x_next_n, forcing), dim=1),
            "x_src": next_step["x_src"].to(device),
        }

    return np.stack(predictions)

In [ ]:
def build_grid(mesh, step):
    x = np.asarray(mesh.meshx[:mesh.npoin2], dtype=np.float64)
    y = np.asarray(mesh.meshy[:mesh.npoin2], dtype=np.float64)
    xi = np.arange(x.min(), x.max() + step, step)
    yi = np.arange(y.min(), y.max() + step, step)
    return np.meshgrid(xi, yi)


def interpolate_depth(mesh, values, grid):
    result, _ = interpolate_on_grid(mesh.tri, np.asarray(values, dtype=np.float64), grid=grid)
    return np.ma.asarray(result)


def grid_csi(prediction, target, mask, threshold):
    pred_values = np.asarray(prediction.filled(np.nan))
    target_values = np.asarray(target.filled(np.nan))
    valid = (~mask) & np.isfinite(pred_values) & np.isfinite(target_values)
    pred_wet = pred_values[valid] >= threshold
    target_wet = target_values[valid] >= threshold
    tp = np.logical_and(pred_wet, target_wet).sum()
    fp = np.logical_and(pred_wet, ~target_wet).sum()
    fn = np.logical_and(~pred_wet, target_wet).sum()
    return float(tp / (tp + fp + fn)) if tp + fp + fn else math.nan


regular_mesh = TelemacFile(REGULAR_MESH_PATH)
fine_mesh = TelemacFile(FINE_MESH_PATH)
GRID = build_grid(fine_mesh, GRID_STEP_M)
DOMAIN_MASK = fine_mesh.tri.get_trifinder()(GRID[0], GRID[1]) == -1

print(f"grid shape={GRID[0].shape} | outside cells={DOMAIN_MASK.sum()}")

In [ ]:
def evaluate_method(method):
    first_dataset = build_dataset(method, EVENTS[0], SEQUENCE_LENGTH)
    model = build_model(method, first_dataset)
    values = {step: [] for step in HORIZONS_STEPS}

    for event in EVENTS:
        dataset = build_dataset(method, event, SEQUENCE_LENGTH)
        fine_sequences = aligned_fine_sequences(event, SEQUENCE_LENGTH)
        sequence_count = min(MAX_SEQUENCES, len(dataset), len(fine_sequences))

        for sequence_index in range(sequence_count):
            predictions = rollout_sequence(model, dataset, sequence_index)
            fine_sequence = fine_sequences[sequence_index]

            for step in HORIZONS_STEPS:
                fine_x, fine_y, _ = unpack_dynamic_sample(fine_sequence[step - 1])
                fine_target = np.asarray(fine_x)[:, 0] + np.asarray(fine_y)[:, 0]
                pred_grid = interpolate_depth(regular_mesh, predictions[step - 1, :, 0], GRID)
                target_grid = interpolate_depth(fine_mesh, fine_target, GRID)
                mask = np.ma.getmaskarray(pred_grid) | np.ma.getmaskarray(target_grid) | DOMAIN_MASK
                values[step].append(grid_csi(pred_grid, target_grid, mask, THRESHOLD_M))

    return {
        step: {
            "mean": float(np.nanmean(values[step])),
            "std": float(np.nanstd(values[step])),
            "count": len(values[step]),
        }
        for step in HORIZONS_STEPS
    }


RESULTS = {method["name"]: evaluate_method(method) for method in METHODS}

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for name in PLOT_METHOD_NAMES:
    mean = np.asarray([RESULTS[name][step]["mean"] for step in HORIZONS_STEPS])
    std = np.asarray([RESULTS[name][step]["std"] for step in HORIZONS_STEPS])
    color = METHOD_BY_NAME[name]["color"]
    ax.plot(HOURS, mean, color=color, linewidth=2, marker="o", label=name)
    ax.fill_between(HOURS, np.clip(mean - std, 0, 1), np.clip(mean + std, 0, 1), color=color, alpha=0.18)

ax.set(
    title=f"Fine-grid CSI at {THRESHOLD_M:.2f} m",
    xlabel="Horizon (hours)",
    ylabel="CSI",
    ylim=(0, 1),
)
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.show()

In [ ]:
def qualitative_payload(method):
    event = next(event for event in EVENTS if event["name"] == QUAL_EVENT_NAME)
    dataset = build_dataset(method, event, QUAL_STEP + 1)
    fine_sequences = aligned_fine_sequences(event, QUAL_STEP + 1)
    model = build_model(method, dataset)
    prediction = rollout_sequence(model, dataset, QUAL_SEQUENCE_INDEX)[QUAL_STEP - 1, :, 0]
    fine_x, fine_y, _ = unpack_dynamic_sample(fine_sequences[QUAL_SEQUENCE_INDEX][QUAL_STEP - 1])
    fine_target = np.asarray(fine_x)[:, 0] + np.asarray(fine_y)[:, 0]

    pred_grid = interpolate_depth(regular_mesh, prediction, GRID)
    target_grid = interpolate_depth(fine_mesh, fine_target, GRID)
    mask = np.ma.getmaskarray(pred_grid) | np.ma.getmaskarray(target_grid) | DOMAIN_MASK
    return {
        "prediction": np.ma.array((pred_grid.filled(np.nan) >= THRESHOLD_M).astype(float), mask=mask),
        "target": np.ma.array((target_grid.filled(np.nan) >= THRESHOLD_M).astype(float), mask=mask),
        "csi": grid_csi(pred_grid, target_grid, mask, THRESHOLD_M),
    }


QUALITATIVE = {name: qualitative_payload(METHOD_BY_NAME[name]) for name in PLOT_METHOD_NAMES}
cmap = ListedColormap(["white", "#1f4e79"])
cmap.set_bad("#d9d9d9")

fig, axes = plt.subplots(1, 1 + len(PLOT_METHOD_NAMES), figsize=(4.5 * (1 + len(PLOT_METHOD_NAMES)), 5))
axes[0].imshow(QUALITATIVE[PLOT_METHOD_NAMES[0]]["target"], origin="lower", cmap=cmap, vmin=0, vmax=1)
axes[0].set_title("Reference")

for ax, name in zip(axes[1:], PLOT_METHOD_NAMES):
    ax.imshow(QUALITATIVE[name]["prediction"], origin="lower", cmap=cmap, vmin=0, vmax=1)
    ax.set_title(f'{name}\nCSI={QUALITATIVE[name]["csi"]:.3f}')

for ax in axes:
    ax.set_axis_off()
fig.suptitle(f"{QUAL_EVENT_NAME} | horizon={QUAL_HOUR} h | threshold={THRESHOLD_M:.2f} m")
fig.tight_layout()
plt.show()